# Week 2 — Fraud Detection
This notebook demonstrates a compact, reproducible workflow for Week 2:
- Load a safe sample of the dataset (defaults to a small, runnable sample),
- Run preprocessing and basic feature engineering,
- Show class distribution, model artifacts and evaluation charts,
- Load and preview saved predictions.

**Notes:** For a full run on the complete dataset, run the `run_week2.py` runner from the repository root (instructions below).

In [ ]:
# Basic imports and paths
import os
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

root = Path.cwd().resolve().parent
data_path = root / 'data' / 'creditcard.csv'
charts_dir = root / 'charts'
models_dir = root / 'models'
outputs_dir = root / 'outputs'
print('data_path ->', data_path)
print('models_dir ->', models_dir)
print('charts_dir ->', charts_dir)
print('outputs_dir ->', outputs_dir)

## Load a runnable sample
To keep the notebook responsive, we load a stratified sample of the full dataset by default.
Set the environment variable `FULL_RUN=1` to force loading the full CSV (not recommended interactively).

In [ ]:
# Load a sample (stratified when possible)
import math
sample_n = int(os.environ.get('NOTEBOOK_SAMPLE_N', 50000))
full_run = os.environ.get('FULL_RUN') == '1'
if not data_path.exists():
    raise FileNotFoundError(f'Expected dataset at {data_path}')
# read only header first to check columns
cols = pd.read_csv(data_path, nrows=0).columns.tolist()
if full_run:
    df = pd.read_csv(data_path)
else:
    # try to stratify by isFraud if present
    if 'isFraud' in cols:
        # read in chunks and sample per class to avoid memory blowup
        chunks = pd.read_csv(data_path, chunksize=200000)
        samples = []
        total = 0
        counts = {}
        for c in chunks:
            total += len(c)
            for k, v in c['isFraud'].value_counts().to_dict().items():
                counts[k] = counts.get(k, 0) + v
            samples.append(c)
        # concat small preview to compute proportions
        df_all = pd.concat(samples, axis=0)
        samples = []
        for cls, cnt in df_all['isFraud'].value_counts().to_dict().items():
            n = max(1, int(round(cnt / len(df_all) * sample_n)))
            sampled = df_all[df_all['isFraud'] == cls].sample(n=n, random_state=42)
            samples.append(sampled)
        df = pd.concat(samples).sample(frac=1, random_state=42).reset_index(drop=True)
    else:
        df = pd.read_csv(data_path, nrows=sample_n)

print('Loaded sample shape:', df.shape)
display(df.head())

In [ ]:
# Import the local preprocessing utilities and run preprocessing on the sample
import sys
sys.path.append(str(root))
from src.preprocessing import preprocess, prepare_train_test, scale_features
df_clean = preprocess(df)
print('After preprocess sample shape:', df_clean.shape)
display(df_clean.head())

In [ ]:
# Show class distribution
if 'isFraud' in df_clean.columns:
    display(df_clean['isFraud'].value_counts(normalize=True))
else:
    print('Target `isFraud` not present in sample — check full dataset or run runner')

## Models, charts and predictions
Below we list available model files and evaluation charts produced by the pipeline. You can open the image files inline.

In [ ]:
# list models and charts
print('Models:')
for p in sorted(models_dir.glob('*.pkl')):
    print(' -', p.name)
print('\nCharts:')
for p in sorted(charts_dir.glob('*.png'))[:20]:
    print(' -', p.name)

# Display a few charts if they exist
sample_charts = ['class_distribution.png', 'roc_curve_random_forest.png', 'confusion_matrix_random_forest.png']
for name in sample_charts:
    f = charts_dir / name
    if f.exists():
        display(Image(str(f)))
    else:
        print('Missing chart:', name)

In [ ]:
# Preview saved predictions (if present)
pred_file = outputs_dir / 'fraud_predictions.csv'
if pred_file.exists():
    preds = pd.read_csv(pred_file, nrows=20)
    display(preds.head())
else:
    print('Predictions file not found:', pred_file)

## Reproduce full pipeline (command line)
From the repository root run:
```
# Activate venv then run:
.venv\Scripts\activate
python Week_2_Fraud_Detection/run_week2.py
```
The runner will subsample automatically for interactive runs; to force a full run, set `FULL_RUN=1` before running (requires sufficient RAM/time).